In [1]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json
from IPython.display import Markdown

load_dotenv(override=True)

True

In [2]:
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)

In [37]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

order_list = [
   "OR-001",
   "OR-002",
   "OR-003"
]

complaint_list = []

In [38]:
def fetch_menu():
    return menu

if order_list:
    def validate_order(order_id):
        if order_id in order_list:
            return {
                "valid": True,
                "message": f"Order ID {order_id} is valid."
            }
        else:
            return {
                "valid": False,
                "message": f"Order ID {order_id} is invalid. Please check and try again."
            }

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },

        "type": "function",
        "function": {
                "name": "validate_order",
                "description": "Validate an order",
                "inputs": {
                    "type": "object",
                    "properties": {
                        "order_id": {
                            "type": "string",
                            "description": "The order ID to validate. OrderId should be like this OR-0000",
                        }
                    },
                    "required": ["order_id"],
                },
                
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [39]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order`** — Call whenever a customer provides an order ID. Use this tool to check if the order ID is valid and retrieve the order details.
        ### Guidelines
        always check the order ID with `validate_order` before logging a complaint. If the order ID is invalid, inform the customer politely and do not log the complaint.
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [40]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order": validate_order
}


def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    completion = llm_client.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools
    )
   
    reply = completion.choices[0].message

    reply_json = reply.model_dump(exclude_none=True)
    reply_json.pop("reasoning", None)            # hide the model's long thinking
    reply_json.pop("reasoning_details", None)
    # print("LLM REPLY:", json.dumps(reply_json, indent=2))

    if reply.tool_calls:                       # model asked for a tool
        messages.append(reply)

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

        completion = llm_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    print(completion.choices[0].message.content)

    return Markdown(completion.choices[0].message.content)

In [41]:
# ask_assistant("i ordered burger but roll was serve d WTH!!! and the bill was made with burger, my name is Sam.")
ask_assistant("i ordered biryani but roll was serve d WTH!!! and the bill was made with burger, my name is asm.")

Hey Asm! I’m sorry to hear about the mix‑up with your order – biryani with a roll and a burger bill sounds frustrating. To help me look into this right away, could you please share your order ID (it should look like OR‑0000)? Once I have that, I can verify the order and sort things out for you.


Hey Asm! I’m sorry to hear about the mix‑up with your order – biryani with a roll and a burger bill sounds frustrating. To help me look into this right away, could you please share your order ID (it should look like OR‑0000)? Once I have that, I can verify the order and sort things out for you.

In [52]:
print(json.dumps(complaint_list))

[{"customer_name": "Sam", "order_id": "OR-1777", "phone": "0322342423", "complaint": "Roll served instead of burger", "status": "Pending"}]
